# Notebook Unificado - Solo Great Expectations (Catalogo Databricks)

In [0]:
%pip install great_expectations pandas openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:

import json
import re
from pathlib import Path

import pandas as pd
import great_expectations as gx
from pyspark.sql import functions as F


In [0]:
%sql
select * from dev_arqanalitica.gobiernodato.regla_negocio_great_expectation limit 100

Pkregla,dominio,Tabla,Campo,Dimension,NombreRegla,descripcion,NombreReglaGreatExpectation,parametros,fecha,responsable,estado,indicadorTabla,path
66,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,name4,Validez,TAXONOMIA APELLIDO PN,"Si el campo FITYP = ""PJ"" el campo NAME4 debe estar nulo o vacio",expectativa_valores_nulos_con_condicion,"{""condicion"":""fityp = 'PJ'"",""columna"":""name4""}",2025-11-19T07:43:53.94791Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
51,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,uwaer,Integridad,Relación moneda y cliente ext.,"si el campo KTOKD (grupo de cuentas) es igual a """"ZEXT"""" entonces el valor del campo WAERS(moneda) debe ser diferente de """"COP"""" para que “CUMPLA”",expectativa_valores_no_permitidos,"{""condicion"":""ktokd = 'ZEXT'"",""columna"":""uwaer"",""valores"":[""COP""]}",2025-11-19T07:43:58.305924Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
71,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,adrnr,Completitud,STD dirección,El campo no puede ser nulo,expectativa_valores_no_nulos,"{""columna"":""adrnr""}",2025-11-10T21:18:10.466849Z,mtrubio@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
76,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,adrnr,Validez,STD dirección,El campo ADRNR no debe iniciar con espacios ni contener caracteres especiales ni tener menos de 8 caracteres,expectativa_valores_con_patron,"{""columna"" : ""adrnr"", ""patron"": ""^(?!s)[A-Za-z0-9]{8,}$""}",2025-11-19T07:43:44.090749Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
81,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,adrnr,Validez,STD dirección,"Si el campo LAND1(pais) es igual a ""CO"" Extraer del campo ADRNR la primera palabra hasta encontrar el primer espacio y si existe en la lista de referencia LR006 ""CUMPLE"" de lo contrario ""NO CUMPLE""",expectativa_primera_palabra_en_referencia_con_condicion,"{""condicion"":""land1 = 'CO'"",""columna"":""adrnr"",""dominio"":""Clientes"",""referencia"":""LR006""}",2025-11-19T07:43:45.1985Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
86,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,ernam,Integridad,Seguridad en creación PSA/VTC,"Si el campo BEGRU(autorización) es igual a ""PSA"" o ""VTC"" el valor del campo ERNAM debe existir en la LR001",expectativa_condicional_tabla_referencia,"{""condicion"": ""begru = 'PSA' OR begru = 'VTC'"", ""columna"": ""ernam"", ""dominio"": ""Clientes"", ""codigo"": ""LR001""}",2025-11-19T07:43:40.500538Z,cmpastrana@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
91,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,ernam,Validez,Seguridad en creación USC,El campo debe corresponder a la lista de referencia definida LR001,expectativa_valores_por_referencia,"{""columna"":""ernam"",""dominio"":""Clientes"",""codigo"":""LR001""}",2025-11-11T03:52:41.843685Z,mtrubio@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
101,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,telf1,Completitud,STD teléfono fijo,El campo no puede ser nulo,expectativa_valores_no_nulos,"{""columna"":""telf1""}",2025-11-11T03:18:46.273483Z,mtrubio@corona.com.co,false,0,abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet
106,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,telf1,Integridad,STD teléfono fijo,"Si el valor del campo TELF1 es ""9999999999"" entonces el campo KTOKD(grupo de cuentas) debe ser igual a cualquiera de estos valores = ""ZCLX"";""ZCON"";""ZDES"";""ZDUM"";""ZEVE"";""ZOBR"";""ZPVT"";""ZTRA"";""ZVEN"" para que ""CUMPLA""",expectativa_condicional_valores_en_lista,"{""condicion"":""telf1 = '9999999999'"",""columna"":""ktokd"",""

In [0]:

# Widgets (Databricks): solo pkregla y resource
# - pkregla: id de regla (OBLIGATORIO)
# - resource: tabla Databricks, ruta parquet o xlsx

RULES_TABLE = "dev_arqanalitica.gobiernodato.regla_negocio_great_expectation"
STRICT_REFERENCE_VALUES = False

# Referencias opcionales para reglas por lista_referencia
REFERENCE_VALUES = {
    # ("Proveedores", "LR001"): ["LGIL", "LMORAV", "LROAJ", "NGALLEGO"],
}
REFERENCE_VALUE_DESC = {
    # ("Clientes", "LR004"): [("57", "ANTIOQUIA"), ("11", "BOGOTA")]
}

if 'dbutils' not in globals() or 'spark' not in globals():
    raise RuntimeError('Este notebook esta dise?ado para Databricks (requiere dbutils y spark).')

dbutils.widgets.text("pkregla", "")
dbutils.widgets.text("resource", "")

PKREGLA = dbutils.widgets.get("pkregla").strip()
RESOURCE = dbutils.widgets.get("resource").strip()

if not PKREGLA:
    raise ValueError("Debes informar pkregla (obligatorio).")

print({"pkregla": PKREGLA, "resource": RESOURCE, "rules_table": RULES_TABLE})


{'pkregla': '116', 'resource': '/Workspace/Users/kagonzalez@corona.com.co/Results/ADR6-CLI.xlsx', 'rules_table': 'dev_arqanalitica.gobiernodato.regla_negocio_great_expectation'}


In [0]:

def sql_condition_to_pandas_query(condition: str, valid_columns=None) -> str:
    """
    Convierte condici?n estilo SQL a query pandas y corrige literales sin comillas.
    Ej: LAND1 = CO -> LAND1 == 'CO'
    """
    cond = (condition or "").strip()
    if not cond or cond.lower() == "true":
        return ""

    cond = re.sub(r"\bAND\b", "and", cond, flags=re.IGNORECASE)
    cond = re.sub(r"\bOR\b", "or", cond, flags=re.IGNORECASE)
    cond = cond.replace("<>", "!=")
    cond = re.sub(r"(?<![<>=!])=(?!=)", "==", cond)

    cols_src = [] if valid_columns is None else list(valid_columns)
    valid_cols = {str(c).upper() for c in cols_src}

    def _needs_quotes(tok: str) -> bool:
        t = tok.strip()
        if not t:
            return False
        if (t.startswith("'") and t.endswith("'")) or (t.startswith('"') and t.endswith('"')):
            return False
        if re.fullmatch(r"[-+]?\d+(\.\d+)?", t):
            return False
        if t.upper() in {"TRUE", "FALSE", "NONE", "NULL"}:
            return False
        if t.upper() in valid_cols:
            return False
        if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", t):
            return True
        return False

    def _quote_rhs(m):
        lhs = m.group(1)
        op = m.group(2)
        rhs = m.group(3).strip()
        if _needs_quotes(rhs):
            rhs = f"'{rhs}'"
        return f"{lhs} {op} {rhs}"

    cond = re.sub(
        r"\b([A-Za-z_][A-Za-z0-9_]*)\s*(==|!=)\s*([^\s\)\(,]+)",
        _quote_rhs,
        cond,
        flags=re.IGNORECASE,
    )

    def _fix_in_clause(m):
        col = m.group(1)
        items = m.group(2)
        parts = [x.strip() for x in items.split(",") if x.strip()]
        fixed = []
        for x in parts:
            fixed.append(f"'{x}'" if _needs_quotes(x) else x)
        return f"{col} in ({', '.join(fixed)})"

    cond = re.sub(
        r"\b([A-Za-z_][A-Za-z0-9_]*)\s+in\s*\(([^\)]*)\)",
        _fix_in_clause,
        cond,
        flags=re.IGNORECASE,
    )

    return cond




def sql_condition_to_spark_expr(condition: str, valid_columns=None) -> str:
    """Convierte una condici?n SQL-like a expresi?n Spark SQL segura para GE."""
    cond = (condition or "").strip()
    if not cond or cond.lower() == "true":
        return ""

    cols_src = [] if valid_columns is None else list(valid_columns)
    valid_cols = {str(c).upper() for c in cols_src}

    cond = cond.replace("<>", "!=")

    def _needs_quotes(tok: str) -> bool:
        t = tok.strip()
        if not t:
            return False
        if (t.startswith("'") and t.endswith("'")) or (t.startswith('"') and t.endswith('"')):
            return False
        if re.fullmatch(r"[-+]?\d+(\.\d+)?", t):
            return False
        if t.upper() in {"NULL", "TRUE", "FALSE"}:
            return False
        if t.upper() in valid_cols:
            return False
        if re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", t):
            return True
        return False

    def _quote_rhs(m):
        lhs = m.group(1)
        op = m.group(2)
        rhs = m.group(3).strip()
        if _needs_quotes(rhs):
            rhs = f"'{rhs}'"
        return f"{lhs} {op} {rhs}"

    cond = re.sub(
        r"\b([A-Za-z_][A-Za-z0-9_]*)\s*(=|!=)\s*([^\s\)\(,]+)",
        _quote_rhs,
        cond,
        flags=re.IGNORECASE,
    )

    return cond

def is_probable_table_name(source_input: str) -> bool:
    s = (source_input or "").strip()
    if not s:
        return False
    if any(s.lower().endswith(ext) for ext in [".xlsx", ".xls", ".parquet", ".csv"]):
        return False
    if "/" in s or "\\" in s or s.lower().startswith(("dbfs:", "abfss:", "s3:", "gs:")):
        return False
    return s.count(".") >= 1




def _resolve_excel_path(path_str: str) -> str:
    # Soporta rutas Workspace y DBFS para lectura con pandas/openpyxl
    p = Path(path_str)
    if p.exists():
        return str(p)

    # Si llega file:/Workspace/... lo normaliza
    if path_str.startswith('file:/'):
        candidate = path_str[len('file:'):]
        if Path(candidate).exists():
            return candidate

    # Compatibilidad dbfs:/ -> /dbfs/
    if path_str.startswith('dbfs:/'):
        candidate = '/dbfs/' + path_str[len('dbfs:/'):]
        if Path(candidate).exists():
            return candidate

    return path_str

def _spark_df_to_pandas(df_spark, columns=None, row_limit=None):
    if columns:
        wanted = [str(c).strip() for c in columns if str(c).strip()]
        if wanted:
            col_map = {str(c).upper(): str(c) for c in df_spark.columns}
            selected = []
            for c in wanted:
                real = col_map.get(c.upper())
                if real and real not in selected:
                    selected.append(real)
            if selected:
                df_spark = df_spark.select(*selected)
    if row_limit is not None:
        df_spark = df_spark.limit(int(row_limit))
    return df_spark.toPandas()


def load_source_to_pandas(resource_input: str, rule_path: str = "", columns=None, row_limit=None):
    try:
        # Evita abortar cuando el dataset completo supera el l?mite default de resultados al driver
        spark.conf.set("spark.driver.maxResultSize", "0")  # noqa: F821
    except Exception:
        pass

    # prioridad: widget resource; si viene vacio usa path de la regla
    src = (resource_input or "").strip() or (rule_path or "").strip()
    if not src:
        raise ValueError("Debes informar widget 'resource' o tener path en la regla")

    low = src.lower()

    if low.endswith((".xlsx", ".xls")):
        # lee Excel en driver
        df_pd = pd.read_excel(_resolve_excel_path(src), sheet_name=0)
        if columns:
            keep = [c for c in columns if str(c) in df_pd.columns]
            if keep:
                df_pd = df_pd[keep]
        if row_limit is not None:
            df_pd = df_pd.head(int(row_limit))
        return df_pd

    if low.endswith(".parquet") or "parquet" in low:
        # parquet en lake/storage
        return _spark_df_to_pandas(spark.read.parquet(src), columns=columns, row_limit=row_limit)  # noqa: F821

    if low.endswith(".csv"):
        return _spark_df_to_pandas(spark.read.option("header", True).csv(src), columns=columns, row_limit=row_limit)  # noqa: F821

    if is_probable_table_name(src):
        return _spark_df_to_pandas(spark.table(src), columns=columns, row_limit=row_limit)  # noqa: F821

    # ultimo intento: spark load por formato detectado
    if src.startswith(("abfss://", "dbfs:/", "s3://", "gs://")):
        try:
            return _spark_df_to_pandas(spark.read.parquet(src), columns=columns, row_limit=row_limit)  # noqa: F821
        except Exception as e:
            raise ValueError(f"No pude leer resource/path como parquet: {src}") from e

    raise ValueError(f"Resource no soportado: {src}")


def load_source_to_spark(resource_input: str, rule_path: str = "", columns=None):
    src = (resource_input or "").strip() or (rule_path or "").strip()
    if not src:
        raise ValueError("Debes informar widget 'resource' o tener path en la regla")

    low = src.lower()
    if low.endswith((".xlsx", ".xls")):
        raise ValueError("Para ejecuci?n masiva usa parquet/csv/tabla, no Excel")

    if low.endswith(".parquet") or "parquet" in low:
        df = spark.read.parquet(src)  # noqa: F821
    elif low.endswith(".csv"):
        df = spark.read.option("header", True).csv(src)  # noqa: F821
    elif is_probable_table_name(src):
        df = spark.table(src)  # noqa: F821
    elif src.startswith(("abfss://", "dbfs:/", "s3://", "gs://")):
        df = spark.read.parquet(src)  # noqa: F821
    else:
        raise ValueError(f"Resource no soportado: {src}")

    if columns:
        wanted = [str(c).strip() for c in columns if str(c).strip()]
        if wanted:
            col_map = {str(c).upper(): str(c) for c in df.columns}
            selected = []
            for c in wanted:
                real = col_map.get(c.upper())
                if real and real not in selected:
                    selected.append(real)
            if selected:
                df = df.select(*selected)

    # normaliza a may?sculas para mantener compatibilidad con par?metros/reglas
    df = df.select(*[F.col(c).alias(str(c).strip().upper()) for c in df.columns])
    return df


def get_ref_values(dominio: str, codigo: str):
    key = (dominio or "", codigo or "")
    if key in REFERENCE_VALUES:
        return list(REFERENCE_VALUES[key]), ""

    try:
        q = f"""
            SELECT DISTINCT valor
            FROM dev_arqanalitica.gobiernodato.lista_referencia
            WHERE dominio = '{dominio}' AND codigo = '{codigo}'
        """
        vals = [r[0] for r in spark.sql(q).collect() if r[0] is not None]  # noqa: F821
        return vals, ""
    except Exception:
        if STRICT_REFERENCE_VALUES:
            raise ValueError(f"Faltan REFERENCE_VALUES para {(dominio, codigo)}")
        return [], f"Referencia {(dominio, codigo)} no configurada"


def get_ref_value_desc(dominio: str, codigo: str):
    key = (dominio or "", codigo or "")
    if key in REFERENCE_VALUE_DESC:
        return list(REFERENCE_VALUE_DESC[key]), ""

    try:
        q = f"""
            SELECT DISTINCT valor, descripcion
            FROM dev_arqanalitica.gobiernodato.lista_referencia
            WHERE dominio = '{dominio}' AND codigo = '{codigo}'
        """
        vals = [(r[0], r[1]) for r in spark.sql(q).collect() if r[0] is not None]  # noqa: F821
        return vals, ""
    except Exception:
        if STRICT_REFERENCE_VALUES:
            raise ValueError(f"Faltan REFERENCE_VALUE_DESC para {(dominio, codigo)}")
        return [], f"Referencia detalle {(dominio, codigo)} no configurada"



SQL_KEYWORDS_COLS = {"AND", "OR", "NOT", "IN", "IS", "NULL", "TRUE", "FALSE", "LIKE", "BETWEEN"}


def _extract_columns_from_condition(condition: str):
    if not condition:
        return []
    toks = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", str(condition).upper())
    return [t for t in toks if t not in SQL_KEYWORDS_COLS]


def get_required_columns(rule_type: str, params: dict):
    cols = set()
    if isinstance(params.get("columna"), str):
        cols.add(params["columna"].strip().upper())

    for c in params.get("columnas", []) or []:
        cols.add(str(c).strip().upper())

    for c in ["referencia", "NAME1", "NAME3", "NAME4"]:
        v = params.get(c)
        if isinstance(v, str) and v.strip():
            cols.add(v.strip().upper())

    t = (rule_type or "").strip().lower()
    if t == "validar_valores_referencia_con_join_tabla":
        cols.update(["REGIO", "TELF1"])

    for cond_key in ["condicion", "condicion_sql"]:
        for c in _extract_columns_from_condition(params.get(cond_key, "")):
            cols.add(c)

    return [c for c in cols if c]


def _to_list(v):
    if isinstance(v, list):
        return v
    if v is None:
        return []
    return [x.strip() for x in str(v).split(',') if x.strip()]


def _add_helper_col(df: pd.DataFrame, base: str, values) -> str:
    name = f"__{base}_{abs(hash(str(values))) % 100000}"
    while name in df.columns:
        name = name + "x"
    return name


In [0]:

# Mapeo completo alineado con Databricks
ALL_EXPECTATION_TYPES = [
    "expectativa_valores_con_patron",
    "expectativa_valor_unico",
    "expectativa_valores_por_referencia_caracteres_con_condicion",
    "expectativa_valores_por_referencia",
    "expectativa_valores_no_nulos",
    "expectativa_con_condicional_y_patron",
    "expectativa_valores_no_permitidos",
    "expectativa_primera_palabra_en_referencia_con_condicion",
    "expectativa_valores_nulos_con_condicion",
    "expectativa_condicional_tabla_referencia",
    "expectativa_condicional_valores_en_lista",
    "expectativa_valores_tabla_equivalencia",
    "expectativa_apellido_nombre_condicional",
    "expectativa_valores_por_referencia_caracteres",
    "expectativa_valores_tipo",
    "expectativa_valores_longitud_entre",
    "expectativa_valores_longitud_igual",
    "expectativa_columna_valores_entre",
    "expectativa_columnas_esperadas",
    "expectativa_combinacion_columnas_unicas",
    "expectativa_valores_distintos_por_referencia",
    "expectativa_cantidad_filas_entre",
    "expectativa_valores_no_vacios",
    "validar_valores_referencia_con_join_tabla",
]


def build_expectation(rule_type: str, params: dict, dataframe: pd.DataFrame):
    t = (rule_type or "").strip().lower()
    warning = ""
    prep = {"df": dataframe.copy()}

    if t not in ALL_EXPECTATION_TYPES:
        raise NotImplementedError(f"Tipo no mapeado: {rule_type}")

    if "columna" in params and isinstance(params["columna"], str):
        params["columna"] = params["columna"].upper().strip()

    cond_sql = params.get("condicion", "")
    cond_pd = sql_condition_to_pandas_query(cond_sql, prep["df"].columns)
    kw_cond = {"row_condition": cond_pd, "condition_parser": "pandas"} if cond_pd else {}

    if t == "expectativa_valores_con_patron":
        exp = gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=params["patron"])
        return exp, prep, warning

    if t == "expectativa_valor_unico":
        exp = gx.expectations.ExpectColumnValuesToBeUnique(column=params["columna"])
        return exp, prep, warning

    if t == "expectativa_valores_por_referencia":
        vals, w = get_ref_values(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        if not vals and not STRICT_REFERENCE_VALUES:
            vals = sorted({str(v).strip() for v in prep['df'][params['columna']].dropna().tolist() if str(v).strip()})
            warning = (warning + "; fallback valores observados").strip('; ')
        exp = gx.expectations.ExpectColumnValuesToBeInSet(column=params["columna"], value_set=vals)
        return exp, prep, warning

    if t == "expectativa_valores_por_referencia_caracteres":
        valores = []
        if params.get("dominio") and params.get("codigo"):
            valores, w = get_ref_values(params.get("dominio", ""), params.get("codigo", ""))
            warning = warning or w
        else:
            valores = _to_list(params.get("referencia"))
        n = int(params.get("caracteres", 1))
        pref = sorted({str(v)[:n] for v in valores if str(v).strip()})
        regex = f"^({'|'.join(pref)})" if pref else r"^$"
        exp = gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=regex)
        return exp, prep, warning

    if t == "expectativa_valores_por_referencia_caracteres_con_condicion":
        pairs, w = get_ref_value_desc(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        n = int(params.get("caracteres", 1))
        descs = sorted({str(d) for _, d in pairs if d is not None})
        pref = sorted({str(v)[:n] for v, _ in pairs if v is not None})
        regex = f"^({'|'.join(pref)})" if pref else r"^$"

        ref_col = str(params.get("referencia", "")).upper().strip()
        base_cond = cond_pd
        in_cond = f"{ref_col} in {descs}" if ref_col and descs else ""
        cond_full = " and ".join([x for x in [base_cond, in_cond] if x])
        kw = {"row_condition": cond_full, "condition_parser": "pandas"} if cond_full else {}

        exp = gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=regex, **kw)
        return exp, prep, warning

    if t == "expectativa_valores_no_nulos":
        exp = gx.expectations.ExpectColumnValuesToNotBeNull(column=params["columna"])
        return exp, prep, warning

    if t == "expectativa_con_condicional_y_patron":
        exp = gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=params["patron"], **kw_cond)
        return exp, prep, warning

    if t == "expectativa_valores_no_permitidos":
        # Normaliza a texto para evitar mismatch de tipos (ej. 0 num?rico vs "0" texto)
        base_col = params["columna"]
        norm_col = f"__GE_NP__{base_col}"
        prep["df"][norm_col] = prep["df"][base_col].fillna("").astype(str).str.strip()
        vals = [str(v).strip() for v in params.get("valores", [])]
        exp = gx.expectations.ExpectColumnValuesToNotBeInSet(column=norm_col, value_set=vals, **kw_cond)
        return exp, prep, warning

    if t == "expectativa_primera_palabra_en_referencia_con_condicion":
        vals, w = get_ref_values(params.get("dominio", ""), params.get("referencia", ""))
        warning = warning or w
        palabras = sorted({str(v).split()[0] for v in vals if str(v).strip()})
        regex = f"^({'|'.join(palabras)})" if palabras else r"^$"
        exp = gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=regex, **kw_cond)
        return exp, prep, warning

    if t == "expectativa_valores_nulos_con_condicion":
        exp = gx.expectations.ExpectColumnValuesToBeNull(column=params["columna"], **kw_cond)
        return exp, prep, warning

    if t == "expectativa_condicional_tabla_referencia":
        vals, w = get_ref_values(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        exp = gx.expectations.ExpectColumnValuesToBeInSet(column=params["columna"], value_set=vals, **kw_cond)
        return exp, prep, warning

    if t == "expectativa_condicional_valores_en_lista":
        exp = gx.expectations.ExpectColumnValuesToBeInSet(column=params["columna"], value_set=params.get("lista", []), **kw_cond)
        return exp, prep, warning

    if t == "expectativa_valores_tabla_equivalencia":
        cols = [str(c).upper().strip() for c in params.get("columnas", [])]
        sep = params.get("separador", "|")
        key_col = _add_helper_col(prep['df'], 'equiv_key', cols)
        prep['df'][key_col] = prep['df'][cols].astype(str).agg(sep.join, axis=1)

        vals, w = get_ref_value_desc(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        allowed = [f"{v}{sep}{d}" for v, d in vals]
        cond2 = sql_condition_to_pandas_query(params.get("condicion", ""), prep["df"].columns)
        kw2 = {"row_condition": cond2, "condition_parser": "pandas"} if cond2 else {}
        exp = gx.expectations.ExpectColumnValuesToBeInSet(column=key_col, value_set=allowed, **kw2)
        return exp, prep, warning

    if t == "expectativa_apellido_nombre_condicional":
        c1 = str(params.get("NAME1", "NAME1")).upper()
        c3 = str(params.get("NAME3", "NAME3")).upper()
        c4 = str(params.get("NAME4", "NAME4")).upper()
        cond2 = sql_condition_to_pandas_query(params.get("condicion_sql", ""), prep["df"].columns)
        helper = _add_helper_col(prep['df'], 'ap_nom_ok', [c1, c3, c4])
        left = prep['df'][c1].fillna('').astype(str).str.strip()
        right = prep['df'][c4].fillna('').astype(str).str.replace(',', ' ', regex=False).str.strip() + ' ' + prep['df'][c3].fillna('').astype(str).str.replace(',', ' ', regex=False).str.strip()
        prep['df'][helper] = (left.str.replace(r'\s+',' ',regex=True).str.strip() == right.str.replace(r'\s+',' ',regex=True).str.strip())
        kw2 = {"row_condition": cond2, "condition_parser": "pandas"} if cond2 else {}
        exp = gx.expectations.ExpectColumnValuesToBeInSet(column=helper, value_set=[True], **kw2)
        return exp, prep, warning

    if t == "expectativa_valores_tipo":
        exp = gx.expectations.ExpectColumnValuesToBeOfType(column=params["columna"], type_=params.get("tipo"))
        return exp, prep, warning

    if t == "expectativa_valores_longitud_entre":
        exp = gx.expectations.ExpectColumnValueLengthsToBeBetween(column=params["columna"], min_value=params.get("min_valor"), max_value=params.get("max_valor"))
        return exp, prep, warning

    if t == "expectativa_valores_longitud_igual":
        exp = gx.expectations.ExpectColumnValueLengthsToEqual(column=params["columna"], value=params.get("longitud"))
        return exp, prep, warning

    if t == "expectativa_columna_valores_entre":
        exp = gx.expectations.ExpectColumnValuesToBeBetween(column=params["columna"], min_value=params.get("min_valor"), max_value=params.get("max_valor"))
        return exp, prep, warning

    if t == "expectativa_columnas_esperadas":
        cols = [str(c).upper().strip() for c in params.get("columnas", [])]
        exp = gx.expectations.ExpectTableColumnsToMatchSet(column_set=cols)
        return exp, prep, warning

    if t == "expectativa_combinacion_columnas_unicas":
        cols = [str(c).upper().strip() for c in params.get("columnas", [])]
        exp = gx.expectations.ExpectCompoundColumnsToBeUnique(column_list=cols)
        return exp, prep, warning

    if t == "expectativa_valores_distintos_por_referencia":
        vals, w = get_ref_values(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        exp = gx.expectations.ExpectColumnValuesToNotBeInSet(column=params["columna"], value_set=vals)
        return exp, prep, warning

    if t == "expectativa_cantidad_filas_entre":
        exp = gx.expectations.ExpectTableRowCountToBeBetween(min_value=params.get("min_valor"), max_value=params.get("max_valor"))
        return exp, prep, warning

    if t == "expectativa_valores_no_vacios":
        exp = gx.expectations.ExpectColumnValuesToNotMatchRegex(column=params["columna"], regex=r"^\s*$")
        return exp, prep, warning

    if t == "validar_valores_referencia_con_join_tabla":
        # Se modela como expectativa booleana sobre una columna helper
        # valida prefijo de TELF1 y matching de REGIO contra tabla de referencia
        ref_pairs, w = get_ref_value_desc(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        n = int(params.get("caracteres", 1))
        cond2 = sql_condition_to_pandas_query(params.get("condicion", ""), prep["df"].columns)

        ref_map = {str(d): str(v)[:n] for v, d in ref_pairs if d is not None and v is not None}
        helper = _add_helper_col(prep['df'], 'join_ref_ok', ref_map)

        regio = prep['df'].get('REGIO', pd.Series('', index=prep['df'].index)).fillna('').astype(str)
        telf1 = prep['df'].get('TELF1', pd.Series('', index=prep['df'].index)).fillna('').astype(str)
        pref = telf1.str[:n]
        prep['df'][helper] = regio.map(ref_map).fillna('__NA__') == pref

        kw2 = {"row_condition": cond2, "condition_parser": "pandas"} if cond2 else {}
        exp = gx.expectations.ExpectColumnValuesToBeInSet(column=helper, value_set=[True], **kw2)
        return exp, prep, warning

    raise NotImplementedError(f"Tipo no soportado: {rule_type}")


def build_expectation_spark(rule_type: str, params: dict, dataframe):
    t = (rule_type or "").strip().lower()
    warning = ""
    prep = {"df": dataframe}

    if t not in ALL_EXPECTATION_TYPES:
        raise NotImplementedError(f"Tipo no mapeado: {rule_type}")

    if "columna" in params and isinstance(params["columna"], str):
        params["columna"] = params["columna"].upper().strip()

    cond_sql = params.get("condicion", "")
    cond_sp = sql_condition_to_spark_expr(cond_sql, prep["df"].columns)
    kw_cond = {"row_condition": cond_sp, "condition_parser": "spark"} if cond_sp else {}

    if t == "expectativa_valores_con_patron":
        return gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=params["patron"]), prep, warning

    if t == "expectativa_valor_unico":
        return gx.expectations.ExpectColumnValuesToBeUnique(column=params["columna"]), prep, warning

    if t == "expectativa_valores_por_referencia":
        vals, w = get_ref_values(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        if not vals and not STRICT_REFERENCE_VALUES:
            warning = (warning + "; fallback sin referencia").strip('; ')
        return gx.expectations.ExpectColumnValuesToBeInSet(column=params["columna"], value_set=vals), prep, warning

    if t == "expectativa_valores_por_referencia_caracteres":
        valores = []
        if params.get("dominio") and params.get("codigo"):
            valores, w = get_ref_values(params.get("dominio", ""), params.get("codigo", ""))
            warning = warning or w
        else:
            valores = _to_list(params.get("referencia"))
        n = int(params.get("caracteres", 1))
        pref = sorted({str(v)[:n] for v in valores if str(v).strip()})
        regex = f"^({'|'.join(pref)})" if pref else r"^$"
        return gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=regex), prep, warning

    if t == "expectativa_valores_por_referencia_caracteres_con_condicion":
        pairs, w = get_ref_value_desc(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        n = int(params.get("caracteres", 1))
        descs = sorted({str(d) for _, d in pairs if d is not None})
        pref = sorted({str(v)[:n] for v, _ in pairs if v is not None})
        regex = f"^({'|'.join(pref)})" if pref else r"^$"

        ref_col = str(params.get("referencia", "")).upper().strip()
        in_vals = ", ".join([repr(x) for x in descs])
        in_cond = f"{ref_col} IN ({in_vals})" if ref_col and descs else ""
        cond_full = " AND ".join([x for x in [cond_sp, in_cond] if x])
        kw = {"row_condition": cond_full, "condition_parser": "spark"} if cond_full else {}

        return gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=regex, **kw), prep, warning

    if t == "expectativa_valores_no_nulos":
        return gx.expectations.ExpectColumnValuesToNotBeNull(column=params["columna"]), prep, warning

    if t == "expectativa_con_condicional_y_patron":
        return gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=params["patron"], **kw_cond), prep, warning

    if t == "expectativa_valores_no_permitidos":
        base_col = params["columna"]
        norm_col = f"__GE_NP__{base_col}"
        prep["df"] = prep["df"].withColumn(norm_col, F.trim(F.coalesce(F.col(base_col).cast("string"), F.lit(""))))
        vals = [str(v).strip() for v in params.get("valores", [])]
        return gx.expectations.ExpectColumnValuesToNotBeInSet(column=norm_col, value_set=vals, **kw_cond), prep, warning

    if t == "expectativa_primera_palabra_en_referencia_con_condicion":
        vals, w = get_ref_values(params.get("dominio", ""), params.get("referencia", ""))
        warning = warning or w
        palabras = sorted({str(v).split()[0] for v in vals if str(v).strip()})
        regex = f"^({'|'.join(palabras)})" if palabras else r"^$"
        return gx.expectations.ExpectColumnValuesToMatchRegex(column=params["columna"], regex=regex, **kw_cond), prep, warning

    if t == "expectativa_valores_nulos_con_condicion":
        return gx.expectations.ExpectColumnValuesToBeNull(column=params["columna"], **kw_cond), prep, warning

    if t == "expectativa_condicional_tabla_referencia":
        vals, w = get_ref_values(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        return gx.expectations.ExpectColumnValuesToBeInSet(column=params["columna"], value_set=vals, **kw_cond), prep, warning

    if t == "expectativa_condicional_valores_en_lista":
        return gx.expectations.ExpectColumnValuesToBeInSet(column=params["columna"], value_set=params.get("lista", []), **kw_cond), prep, warning

    if t == "expectativa_valores_tabla_equivalencia":
        cols = [str(c).upper().strip() for c in params.get("columnas", [])]
        sep = params.get("separador", "|")
        key_col = _add_helper_col(pd.DataFrame(columns=[]), 'equiv_key', cols)
        exprs = [F.coalesce(F.col(c).cast("string"), F.lit("")) for c in cols]
        prep["df"] = prep["df"].withColumn(key_col, F.concat_ws(sep, *exprs))

        vals, w = get_ref_value_desc(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        allowed = [f"{v}{sep}{d}" for v, d in vals]
        cond2 = sql_condition_to_spark_expr(params.get("condicion", ""), prep["df"].columns)
        kw2 = {"row_condition": cond2, "condition_parser": "spark"} if cond2 else {}
        return gx.expectations.ExpectColumnValuesToBeInSet(column=key_col, value_set=allowed, **kw2), prep, warning

    if t == "expectativa_apellido_nombre_condicional":
        c1 = str(params.get("NAME1", "NAME1")).upper()
        c3 = str(params.get("NAME3", "NAME3")).upper()
        c4 = str(params.get("NAME4", "NAME4")).upper()
        cond2 = sql_condition_to_spark_expr(params.get("condicion_sql", ""), prep["df"].columns)
        helper = _add_helper_col(pd.DataFrame(columns=[]), 'ap_nom_ok', [c1, c3, c4])

        left = F.regexp_replace(F.trim(F.coalesce(F.col(c1).cast("string"), F.lit(""))), r"\s+", " ")
        right_c4 = F.regexp_replace(F.trim(F.regexp_replace(F.coalesce(F.col(c4).cast("string"), F.lit("")), ",", " ")), r"\s+", " ")
        right_c3 = F.regexp_replace(F.trim(F.regexp_replace(F.coalesce(F.col(c3).cast("string"), F.lit("")), ",", " ")), r"\s+", " ")
        right = F.regexp_replace(F.trim(F.concat_ws(" ", right_c4, right_c3)), r"\s+", " ")

        prep["df"] = prep["df"].withColumn(helper, left == right)
        kw2 = {"row_condition": cond2, "condition_parser": "spark"} if cond2 else {}
        return gx.expectations.ExpectColumnValuesToBeInSet(column=helper, value_set=[True], **kw2), prep, warning

    if t == "expectativa_valores_tipo":
        return gx.expectations.ExpectColumnValuesToBeOfType(column=params["columna"], type_=params.get("tipo")), prep, warning

    if t == "expectativa_valores_longitud_entre":
        return gx.expectations.ExpectColumnValueLengthsToBeBetween(column=params["columna"], min_value=params.get("min_valor"), max_value=params.get("max_valor")), prep, warning

    if t == "expectativa_valores_longitud_igual":
        return gx.expectations.ExpectColumnValueLengthsToEqual(column=params["columna"], value=params.get("longitud")), prep, warning

    if t == "expectativa_columna_valores_entre":
        return gx.expectations.ExpectColumnValuesToBeBetween(column=params["columna"], min_value=params.get("min_valor"), max_value=params.get("max_valor")), prep, warning

    if t == "expectativa_columnas_esperadas":
        cols = [str(c).upper().strip() for c in params.get("columnas", [])]
        return gx.expectations.ExpectTableColumnsToMatchSet(column_set=cols), prep, warning

    if t == "expectativa_combinacion_columnas_unicas":
        cols = [str(c).upper().strip() for c in params.get("columnas", [])]
        return gx.expectations.ExpectCompoundColumnsToBeUnique(column_list=cols), prep, warning

    if t == "expectativa_valores_distintos_por_referencia":
        vals, w = get_ref_values(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        return gx.expectations.ExpectColumnValuesToNotBeInSet(column=params["columna"], value_set=vals), prep, warning

    if t == "expectativa_cantidad_filas_entre":
        return gx.expectations.ExpectTableRowCountToBeBetween(min_value=params.get("min_valor"), max_value=params.get("max_valor")), prep, warning

    if t == "expectativa_valores_no_vacios":
        return gx.expectations.ExpectColumnValuesToNotMatchRegex(column=params["columna"], regex=r"^\s*$"), prep, warning

    if t == "validar_valores_referencia_con_join_tabla":
        ref_pairs, w = get_ref_value_desc(params.get("dominio", ""), params.get("codigo", ""))
        warning = warning or w
        n = int(params.get("caracteres", 1))
        cond2 = sql_condition_to_spark_expr(params.get("condicion", ""), prep["df"].columns)

        ref_map = {str(d): str(v)[:n] for v, d in ref_pairs if d is not None and v is not None}
        helper = _add_helper_col(pd.DataFrame(columns=[]), 'join_ref_ok', ref_map)

        regio = F.coalesce(F.col('REGIO').cast('string'), F.lit(''))
        pref = F.substring(F.coalesce(F.col('TELF1').cast('string'), F.lit('')), 1, n)

        if ref_map:
            map_expr_items = []
            for k, v in ref_map.items():
                map_expr_items.extend([F.lit(k), F.lit(v)])
            map_expr = F.create_map(*map_expr_items)
            expected = F.element_at(map_expr, regio)
            prep["df"] = prep["df"].withColumn(helper, F.coalesce(expected, F.lit('__NA__')) == pref)
        else:
            prep["df"] = prep["df"].withColumn(helper, F.lit(False))

        kw2 = {"row_condition": cond2, "condition_parser": "spark"} if cond2 else {}
        return gx.expectations.ExpectColumnValuesToBeInSet(column=helper, value_set=[True], **kw2), prep, warning

    raise NotImplementedError(f"Tipo no soportado: {rule_type}")


In [0]:

def ge_validate_dataframe(df: pd.DataFrame, expectation, suite_name: str):
    context = gx.get_context(mode="ephemeral")
    ds_name = f"pandas_ds_{suite_name}"
    datasource = context.data_sources.add_pandas(name=ds_name)
    asset = datasource.add_dataframe_asset(name="asset")
    batch = asset.add_batch_definition_whole_dataframe(name="batch")

    suite = gx.ExpectationSuite(name=suite_name)
    suite.add_expectation(expectation)
    context.suites.add_or_update(suite)

    vd = gx.ValidationDefinition(data=batch, suite=suite, name=f"vd_{suite_name}")
    context.validation_definitions.add(vd)

    res = vd.run(batch_parameters={"dataframe": df})
    return res.to_json_dict() if hasattr(res, 'to_json_dict') else res.validation_result.to_json_dict()


def ge_validate_spark_dataframe(df, expectation, suite_name: str):
    context = gx.get_context(mode="ephemeral")
    ds_name = f"spark_ds_{suite_name}"
    datasource = context.data_sources.add_spark(name=ds_name)
    asset = datasource.add_dataframe_asset(name="asset")
    batch = asset.add_batch_definition_whole_dataframe(name="batch")

    rf = {"result_format": "SUMMARY", "partial_unexpected_count": 0, "include_unexpected_rows": False}
    if hasattr(expectation, "kwargs") and isinstance(getattr(expectation, "kwargs"), dict):
        expectation.kwargs["result_format"] = rf
    elif hasattr(expectation, "result_format"):
        try:
            expectation.result_format = rf
        except Exception:
            pass

    suite = gx.ExpectationSuite(name=suite_name)
    suite.add_expectation(expectation)
    context.suites.add_or_update(suite)

    vd = gx.ValidationDefinition(data=batch, suite=suite, name=f"vd_{suite_name}")
    context.validation_definitions.add(vd)

    res = vd.run(batch_parameters={"dataframe": df})
    return res.to_json_dict() if hasattr(res, 'to_json_dict') else res.validation_result.to_json_dict()


def summarize(rule_type: str, validation_json: dict, total_rows: int, df_for_fallback: pd.DataFrame):
    r0 = (validation_json.get('results') or [{}])[0]
    kwargs = (r0.get('expectation_config') or {}).get('kwargs', {}) or {}
    result = r0.get('result', {}) or {}

    applicable = result.get('element_count')
    rc = kwargs.get('row_condition')
    if isinstance(rc, dict):
        rc = rc.get('pass_through_filter')
    if applicable is None:
        if rc and df_for_fallback is not None and hasattr(df_for_fallback, "query"):
            try:
                applicable = int(df_for_fallback.query(rc).shape[0])
            except Exception:
                applicable = total_rows
        else:
            applicable = total_rows
    applicable = int(applicable)

    unexpected = int(result.get('unexpected_count') or 0)
    missing = int(result.get('missing_count') or 0)

    rt = (rule_type or '').strip().lower()
    count_missing_as_fail = rt in {
        'expectativa_condicional_valores_en_lista',
        'expectativa_valores_por_referencia',
        'expectativa_condicional_tabla_referencia',
    }

    no_cumple = min(unexpected + (missing if count_missing_as_fail else 0), applicable)
    cumple = max(applicable - no_cumple, 0)
    no_aplica = max(total_rows - applicable, 0)
    pct = round((cumple / applicable) * 100, 2) if applicable > 0 else 0.0

    return {
        'total': total_rows,
        'cumple': cumple,
        'no_cumple': no_cumple,
        'no_aplica': no_aplica,
        'pct_calidad': pct,
        'ge_success': bool(r0.get('success')),
        'ge_unexpected_count': unexpected,
        'ge_missing_count': missing,
        'ge_row_condition': rc or '',
    }


In [0]:

def load_rule(pkregla: str):
    pk = str(pkregla).replace("'", "''")
    q = f"SELECT * FROM {RULES_TABLE} WHERE CAST(Pkregla AS STRING) = '{pk}'"
    pdf = spark.sql(q).limit(1).toPandas().fillna('')  # noqa: F821

    if pdf.empty:
        raise ValueError(f"No existe Pkregla={pkregla} en {RULES_TABLE}")

    if len(pdf) > 1:
        print(f"Advertencia: se encontraron {len(pdf)} filas para pkregla={pkregla}; se toma la primera")

    return pdf.iloc[[0]].copy()


run_rules = load_rule(PKREGLA)
run_rules['Pkregla'] = run_rules['Pkregla'].astype(str).str.strip()
run_rules['Campo'] = run_rules['Campo'].astype(str).str.upper().str.strip()

print(f"Regla cargada: {len(run_rules)}")
print(run_rules[['Pkregla','dominio','Tabla','Campo','NombreReglaGreatExpectation','path']].to_string(index=False))

outputs = []
for _, r in run_rules.iterrows():
    params = json.loads((r.get('parametros') or '{}').strip() or '{}')

    needed_cols = get_required_columns(r['NombreReglaGreatExpectation'], params)
    needed_cols = sorted(set(needed_cols + [str(r.get('Campo', '')).strip().upper()]))
    df = load_source_to_spark(RESOURCE, rule_path=str(r.get('path','')), columns=needed_cols)

    try:
        expectation, prep, warning = build_expectation_spark(r['NombreReglaGreatExpectation'], params, df)
        dfr = prep['df']
        validation_json = ge_validate_spark_dataframe(dfr, expectation, suite_name=f"suite_{r['Pkregla']}")
        total_rows = int(dfr.count())
        met = summarize(r['NombreReglaGreatExpectation'], validation_json, total_rows, None)
        estado = 'CUMPLE' if met['no_cumple'] == 0 else 'NO CUMPLE'
    except Exception as e:
        warning = f'ERROR: {type(e).__name__}: {e}'
        total_rows = int(df.count())
        met = {'total': total_rows, 'cumple': 0, 'no_cumple': 0, 'no_aplica': total_rows, 'pct_calidad': 0.0,
               'ge_success': False, 'ge_unexpected_count': 0, 'ge_missing_count': 0, 'ge_row_condition': ''}
        estado = 'ERROR'

    outputs.append({
        'Pkregla': r['Pkregla'],
        'dominio': r.get('dominio',''),
        'Tabla': r['Tabla'],
        'Campo': r['Campo'],
        'Dimension': r.get('Dimension',''),
        'NombreRegla': r['NombreRegla'],
        'descripcion': r.get('descripcion',''),
        'NombreReglaGreatExpectation': r['NombreReglaGreatExpectation'],
        'path': r.get('path',''),
        **met,
        'estado': estado,
        'warning': warning,
    })

resultado = pd.DataFrame(outputs)
resultado


Regla cargada: 1
Pkregla        dominio             Tabla Campo          NombreReglaGreatExpectation                                                                                         path
    116 DP_CL-Clientes BR-SAP.SD.KNA1_CI TELF1 expectativa_con_condicional_y_patron abfss://bronce-zone@saarqanaliticaprd.dfs.core.windows.net/SAP/CI/DM/SD/KNA1/*/*/*/*.parquet


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

,Pkregla,dominio,Tabla,Campo,Dimension,NombreRegla,descripcion,NombreReglaGreatExpectation,path,total,cumple,no_cumple,no_aplica,pct_calidad,ge_success,ge_unexpected_count,ge_missing_count,ge_row_condition,estado,warning
0,116,DP_CL-Clientes,BR-SAP.SD.KNA1_CI,TELF1,Integridad,STD teléfono fijo,Si el campo pais LAND1 es igual CO entonces e...,expectativa_con_condicional_y_patron,abfss://bronce-zone@saarqanaliticaprd.dfs.core...,100,100,0,0,100.0,False,0,0,LAND1 == CO,CUMPLE,


In [0]:
print(resultado[['Pkregla','NombreReglaGreatExpectation','estado','total','cumple','no_cumple','no_aplica','warning']].to_string(index=False))

Pkregla          NombreReglaGreatExpectation estado  total  cumple  no_cumple  no_aplica warning
    116 expectativa_con_condicional_y_patron CUMPLE    100     100          0          0        


In [0]:
# Resumen final (1 fila) para la regla ejecutada
import pandas as pd

if 'resultado' not in globals() or len(resultado) == 0:
    raise ValueError("No hay resultado. Ejecuta primero la celda de validaciÃ³n.")

row = resultado.iloc[0].copy()

# Ajuste de negocio para expectativa_valores_no_permitidos:
# no_cumple = unexpected_count (missing NO cuenta como no_cumple)
if str(row.get("NombreReglaGreatExpectation", "")).strip().lower() == "expectativa_valores_no_permitidos":
    total = int(row["total"])
    no_aplica = int(row["no_aplica"])
    no_cumple = int(row.get("ge_unexpected_count", row["no_cumple"]))
    cumple = (total - no_aplica) - no_cumple
    pct = round((cumple / (total - no_aplica)) * 100, 2) if (total - no_aplica) > 0 else 0.0
else:
    total = int(row["total"])
    cumple = int(row["cumple"])
    no_cumple = int(row["no_cumple"])
    no_aplica = int(row["no_aplica"])
    pct = float(row["pct_calidad"])

resumen_final = pd.DataFrame([{
    "total": total,
    "cumple": cumple,
    "no_cumple": no_cumple,
    "no_aplica": no_aplica,
    "pct_calidad": pct
}])

display(resumen_final)


,total,cumple,no_cumple,no_aplica,pct_calidad
0,100,100,0,0,100.0


In [0]:
# Ver contenido del resource (parquet / xlsx / tabla Databricks) con display

# Usa RESOURCE del widget; si viene vacÃ­o usa el path de la regla cargada
src = (RESOURCE or "").strip()
if not src:
    if "run_rules" in globals() and len(run_rules) > 0:
        src = str(run_rules.iloc[0].get("path", "")).strip()
    else:
        raise ValueError("No hay RESOURCE ni regla cargada para tomar path.")

if not src:
    raise ValueError("No se encontrÃ³ fuente para mostrar.")

print(f"Fuente usada: {src}")

# Carga segÃºn tipo
low = src.lower()
if low.endswith((".xlsx", ".xls")):
    pdf = pd.read_excel(src, sheet_name=0)
elif low.endswith(".parquet") or "parquet" in low:
    pdf = spark.read.parquet(src).toPandas()
elif "." in src and not any(x in src for x in ["/", "\\", "dbfs:", "abfss:", "s3://", "gs://"]):
    # tabla databricks tipo catalog.schema.table
    pdf = spark.table(src).toPandas()
else:
    # intento final (por ejemplo rutas lake sin extensiÃ³n clara)
    pdf = spark.read.parquet(src).toPandas()

# Normaliza columnas y muestra
pdf.columns = [str(c).strip().upper() for c in pdf.columns]

display(pd.DataFrame([{
    "source": src,
    "rows": len(pdf),
    "columns": len(pdf.columns)
}]))

display(pdf.head(100))


Fuente usada: /Workspace/Users/kagonzalez@corona.com.co/Results/ADR6-CLI.xlsx


,source,rows,columns
0,/Workspace/Users/kagonzalez@corona.com.co/Resu...,100,193


,AERUNID,AERECNO,MANDT,KUNNR,LAND1,NAME1,NAME2,ORT01,PSTLZ,REGIO,SORTL,STRAS,TELF1,TELFX,XCPDK,ADRNR,MCOD1,MCOD2,MCOD3,ANRED,AUFSD,BAHNE,BAHNS,BBBNR,BBSNR,BEGRU,BRSCH,BUBKZ,DATLT,ERDAT,ERNAM,EXABL,FAKSD,FISKN,KNAZK,KNRZA,KONZS,KTOKD,KUKLA,LIFNR,...,CRTN,ICMSTAXPAY,INDTYP,TDT,COMSIZE,DECREGPC,_VSO_R_PALHGT,_VSO_R_PAL_UL,_VSO_R_PK_MAT,_VSO_R_MATPAL,_VSO_R_I_NO_LYR,_VSO_R_ONE_MAT,_VSO_R_ONE_SORT,_VSO_R_ULD_SIDE,_VSO_R_LOAD_PREF,_VSO_R_DPOINT,ALC,PMT_OFFICE,FEE_SCHEDULE,DUNS,DUNS4,PSOFG,PSOIS,PSON1,PSON2,PSON3,PSOVN,PSOTL,PSOHS,PSOST,PSOO1,PSOO2,PSOO3,PSOO4,PSOO5,ZZTRADT,ZZPTORG,ZZNRHIJ,ZZCATEGORIA,OPFLAG
0,93,1000070,300,1000499958,CO,BARRASA ELADIO,NaN,BOGOTA D.C.,11001,11,72042370,CR 105 B 69 B 13 CA CALLE 80 87 47,0,0,NaN,1230871,BARRASA ELADIO,NaN,BOGOTA D.C.,Señor (es),NaN,NaN,NaN,0,0,ZTIE,NaN,0,NaN,2022-07-16,JMONSALVEH,X,NaN,NaN,NaN,NaN,NaN,ZTIE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,A63N,0,2,NaN
1,93,1000091,300,1000499962,CO,BAEZ RAFAEL,NaN,BOGOTA D.C.,11001,11,7211322,CR 94 22 A 90 INTE 6 AP 603 PASEO D,0,0,NaN,1230879,BAEZ RAFAEL,NaN,BOGOTA D.C.,Señor (es),NaN,NaN,NaN,0,0,ZTIE,NaN,0,NaN,2022-07-16,JMONSALVEH,X,NaN,NaN,NaN,NaN,NaN,ZTIE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,A63N,0,2,NaN
2,93,1000069,300,1000500015,CO,MURCIA HELADIO ORTEGON,NaN,BOGOTA D.C.,11001,11,7306532,DG 49 SUR 51 50 CA,0,0,NaN,1230985,MURCIA HELADIO ORTEGON,NaN,BOGOTA D.C.,Señor (es),NaN,NaN,NaN,0,0,ZTIE,NaN,0,NaN,2022-07-16,JMONSALVEH,X,NaN,NaN,NaN,NaN,NaN,ZTIE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,A63N,0,2,NaN
3,93,1000024,300,1000500031,CO,BUSTOS MIGUEL ANGEL,NaN,BOGOTA D.C.,11001,11,7317816,CR 96 159 A 18 CA,0,0,NaN,1231017,BUSTOS MIGUEL ANGEL,NaN,BOGOTA D.C.,Señor (es),NaN,NaN,NaN,0,0,ZTIE,NaN,0,NaN,2022-07-16,JMONSALVEH,X,NaN,NaN,NaN,NaN,NaN,ZTIE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,A33N,0,2,NaN
4,93,1000092,300,1000500044,CO,BARDA PABON JAIRO ANTONIO,NaN,BOGOTA D.C.,11001,11,7335058,CL 70 C 111 A 69 CA,0,0,NaN,1231043,BARDA PABON JAIRO ANTONIO,NaN,BOGOTA D.C.,Señor (es),NaN,NaN,NaN,0,0,ZTIE,NaN,0,NaN,2022-07-16,JMONSALVEH,X,NaN,NaN,NaN,NaN,NaN,ZTIE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,A63N,0,2,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,93,1000088,300,1000505858,CO,DE ECHEVERRY PATRICIA CARRILLO,NaN,BOGOTA D.C.,11001,11,35499792,CL 125 18A 05 AP 203,0,0,NaN,1256227,DE ECHEVERRY PATRICIA CAR,NaN,BOGOTA D.C.,Señor (es),NaN,NaN,NaN,0,0,ZTIE,NaN,0,NaN,2022-07-18,JMONSALVEH,X,NaN,NaN,NaN,NaN,NaN,ZTIE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,A63N,0,2,NaN
96,93,1000066,300,1000506581,CO,SILVA ACEVEDO MARIA HELDA,NaN,BOGOTA D.C.,11001,11,51579472,AV CARACAS 45C 37 SUR OF AV.CARACAS,0,0,NaN,1257669,SILVA ACEVEDO MARIA HELDA,NaN,BOGOTA D.C.,Señor (es),NaN,NaN,NaN,0,0,ZTIE,NaN,0,NaN,2022-07-18,JMONSALVEH,X,NaN,NaN,NaN,NaN,NaN,ZTIE,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,A63N,0,2,NaN
97,93,1000095,300,1000507990,CO,VARGAS NELSON,NaN,BOGOTA D.C.,11001,11,74369900,CL 26 86 85 CA,0,0,NaN,1260473,VARGAS NELSON,NaN,BOGOTA 

In [0]:
# Celda final: JSON detallado por registro (CUMPLE / NO_CUMPLE / NO_APLICA)

import json
import re
import warnings
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=DeprecationWarning)

if "resultado" not in globals() or resultado.empty:
    raise ValueError("No existe 'resultado' o est? vac?o. Ejecuta primero la celda de validaci?n.")

MAX_ROWS_JSON = 2000
SQL_KEYWORDS = {"AND", "OR", "NOT", "IN", "IS", "NULL", "TRUE", "FALSE", "LIKE", "BETWEEN"}

def _extract_cond_cols(cond: str):
    if not cond:
        return []
    toks = re.findall(r"[A-Za-z_][A-Za-z0-9_]*", str(cond).upper())
    return [t for t in toks if t not in SQL_KEYWORDS]

def _to_json_value(v):
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return v.isoformat() if isinstance(v, pd.Timestamp) else v

def _normalize_condition(cond):
    if isinstance(cond, dict):
        return cond.get("pass_through_filter") or cond.get("row_condition") or ""
    return cond or ""

def _align_condition_columns(cond_: str, columns):
    if not cond_:
        return cond_
    cols_map = {str(c).upper(): str(c) for c in columns}
    keywords = SQL_KEYWORDS | {"AND", "OR", "NOT", "IN", "IS", "NULL", "TRUE", "FALSE"}

    def repl(m):
        tok = m.group(0)
        up = tok.upper()
        if up in keywords:
            return tok
        return cols_map.get(up, tok)

    return re.sub(r"[A-Za-z_][A-Za-z0-9_]*", repl, str(cond_))


def _safe_query(df_, cond_):
    if not cond_:
        return df_
    cond_aligned = _align_condition_columns(cond_, df_.columns)
    try:
        return df_.query(cond_aligned)
    except Exception:
        # Si la condici?n no se puede evaluar, no marcar todo como aplicable
        return df_.iloc[0:0]

def _motivo_no_cumple(exp_name: str, campo: str, row: pd.Series, params: dict):
    exp = (exp_name or "").strip().lower()
    val = "" if campo not in row.index or pd.isna(row.get(campo)) else str(row.get(campo)).strip()

    if exp in {"expectativa_con_condicional_y_patron", "expectativa_valores_con_patron"}:
        patron = str(params.get("patron", ""))
        motivos = []
        if val == "":
            motivos.append("vac?o o nulo")
        if " " in val:
            motivos.append("contiene espacios")
        if "\\d" in patron and not val.isdigit():
            motivos.append("contiene caracteres no num?ricos")
        if not motivos:
            motivos.append(f"no coincide con patr?n {patron}")
        return "; ".join(motivos)

    if exp == "expectativa_condicional_valores_en_lista":
        permitidos = {str(x).strip() for x in params.get("lista", [])}
        return f"valor '{val}' fuera de lista permitida {sorted(permitidos)}"

    if exp == "expectativa_valores_no_permitidos":
        no_permitidos = {str(x).strip() for x in params.get("valores", [])}
        return f"valor '{val}' est? en lista no permitida {sorted(no_permitidos)}"

    if exp == "expectativa_valores_no_nulos":
        return "valor nulo"

    if exp == "expectativa_valores_nulos_con_condicion":
        return f"debe ser nulo y lleg? '{val}'"

    return "incumple la expectativa"

def _build_fail_mask(exp_name: str, app_df: pd.DataFrame, campo: str, params: dict):
    exp = (exp_name or "").strip().lower()
    if campo not in app_df.columns:
        return pd.Series([False] * len(app_df), index=app_df.index)

    s = app_df[campo]
    s_txt = s.fillna("").astype(str).str.strip()

    if exp == "expectativa_condicional_valores_en_lista":
        permitidos = {str(x).strip() for x in params.get("lista", [])}
        return (~s_txt.isin(permitidos)) | s_txt.eq("")

    if exp in {"expectativa_con_condicional_y_patron", "expectativa_valores_con_patron"}:
        patron = str(params.get("patron", ""))
        return ~s_txt.str.contains(patron, regex=True, na=False)

    if exp == "expectativa_valores_no_permitidos":
        no_permitidos = {str(x).strip() for x in params.get("valores", [])}
        return s_txt.isin(no_permitidos)

    if exp == "expectativa_valores_no_nulos":
        return s.isna()

    if exp == "expectativa_valores_nulos_con_condicion":
        return ~s.isna()

    return pd.Series([False] * len(app_df), index=app_df.index)

def _detalle_regla(rule_row: dict, max_rows: int = 2000):
    pk = str(rule_row.get("Pkregla", "")).strip()

    if str(rule_row.get("estado", "")).strip().upper() == "ERROR":
        return {
            "Pkregla": pk,
            "Tabla": rule_row.get("Tabla"),
            "Campo": rule_row.get("Campo"),
            "expectativa": rule_row.get("NombreReglaGreatExpectation"),
            "estado_regla": "ERROR",
            "warning": rule_row.get("warning", ""),
            "resumen": {
                "total": int(rule_row.get("total", 0) or 0),
                "cumple": int(rule_row.get("cumple", 0) or 0),
                "no_cumple": int(rule_row.get("no_cumple", 0) or 0),
                "no_aplica": int(rule_row.get("no_aplica", 0) or 0),
            },
            "registros": []
        }

    rr = load_rule(pk).iloc[0]
    params = json.loads((rr.get("parametros") or "{}").strip() or "{}")

    needed_cols = get_required_columns(rr["NombreReglaGreatExpectation"], params)
    needed_cols = sorted(set(needed_cols + [str(rr.get("Campo", "")).strip().upper()]))
    df = load_source_to_pandas(RESOURCE, rule_path=str(rr.get("path", "")), columns=needed_cols, row_limit=max_rows).copy()
    df.columns = [str(c).strip().upper() for c in df.columns]

    expectation, prep, _warning = build_expectation(rr["NombreReglaGreatExpectation"], params, df)
    dfr = prep["df"].copy()

    rf = {"result_format": "SUMMARY", "partial_unexpected_count": min(max_rows, 200), "include_unexpected_rows": False}
    if hasattr(expectation, "kwargs") and isinstance(getattr(expectation, "kwargs"), dict):
        expectation.kwargs["result_format"] = rf
    elif hasattr(expectation, "result_format"):
        try:
            expectation.result_format = rf
        except Exception:
            pass

    validation_json = ge_validate_dataframe(dfr, expectation, suite_name=f"suite_detail_{pk}")
    r0 = (validation_json.get("results") or [{}])[0]
    res = r0.get("result", {}) or {}
    kwargs = (r0.get("expectation_config") or {}).get("kwargs", {}) or {}

    exp_name = str(rr.get("NombreReglaGreatExpectation", "")).strip().lower()
    # Para no_permitidos se valida sobre columna normalizada creada en build_expectation
    campo = str(rr.get("Campo", "")).strip().upper()
    if exp_name == "expectativa_valores_no_permitidos" and f"__GE_NP__{campo}" in dfr.columns:
        campo = f"__GE_NP__{campo}"
    campo_real = str(rr.get("Campo", "")).strip().upper()

    cond_raw = kwargs.get("row_condition") or params.get("condicion") or params.get("condicion_sql") or ""
    cond = _normalize_condition(cond_raw)

    app_df = _safe_query(dfr, cond)
    applicable_idx = set(app_df.index.tolist())

    idxs = res.get("unexpected_index_list") or []
    fail_idx = set(idxs) if idxs else set()

    # Fallback local para cubrir casos donde GE no detecta fail por tipos/serializaci?n
    need_fallback = (
        not fail_idx and campo in dfr.columns and (
            int(rule_row.get("no_cumple", 0) or 0) > 0
            or exp_name == "expectativa_valores_no_permitidos"
        )
    )
    if need_fallback:
        fail_mask = _build_fail_mask(exp_name, app_df, campo, params)
        try:
            fail_idx = set(app_df.loc[fail_mask].index.tolist())
        except Exception:
            fail_idx = set()

    cond_cols = [c for c in _extract_cond_cols(cond) if c in dfr.columns]
    id_cols = [c for c in ["LIFNR", "KUNNR", "MATNR"] if c in dfr.columns]
    ctx_cols = list(dict.fromkeys(id_cols + cond_cols))

    registros = []
    for i, (idx, row) in enumerate(dfr.iterrows()):
        if i >= max_rows:
            break

        if idx not in applicable_idx:
            estado = "NO_APLICA"
            motivo = f"No cumple condici?n: {cond}" if cond else "No aplica por condici?n"
        elif idx in fail_idx:
            estado = "NO_CUMPLE"
            motivo = _motivo_no_cumple(exp_name, campo, row, params)
        else:
            estado = "CUMPLE"
            motivo = "Cumple condici?n y expectativa"

        registros.append({
            "row_index": int(idx) if str(idx).isdigit() else str(idx),
            "estado": estado,
            "motivo": motivo,
            "valor_campo": _to_json_value(row.get(campo_real)) if campo_real in dfr.columns else None,
            "contexto": {c: _to_json_value(row.get(c)) for c in ctx_cols}
        })

    c_cumple = sum(1 for x in registros if x["estado"] == "CUMPLE")
    c_nocumple = sum(1 for x in registros if x["estado"] == "NO_CUMPLE")
    c_noaplica = sum(1 for x in registros if x["estado"] == "NO_APLICA")

    return {
        "Pkregla": pk,
        "Tabla": rr.get("Tabla"),
        "Campo": campo_real,
        "expectativa": rr.get("NombreReglaGreatExpectation"),
        "estado_regla": rule_row.get("estado"),
        "condicion_sql": cond,
        "resumen": {
            "total": int(len(dfr)),
            "cumple": int(c_cumple),
            "no_cumple": int(c_nocumple),
            "no_aplica": int(c_noaplica),
            "registros_mostrados": int(len(registros)),
            "max_rows_json": int(max_rows),
        },
        "registros": registros,
    }

rules_input = resultado.to_dict(orient="records")
rules_detailed = [_detalle_regla(r, max_rows=MAX_ROWS_JSON) for r in rules_input]

payload_detailed = {
    "totals": {
        "rules": int(len(resultado)),
        "failed": int((resultado["estado"] == "NO CUMPLE").sum()),
        "errors": int((resultado["estado"] == "ERROR").sum()),
        "passed": int((resultado["estado"] == "CUMPLE").sum()),
        "with_no_aplica": int((resultado["no_aplica"] > 0).sum()) if "no_aplica" in resultado.columns else 0,
    },
    "rules_detailed": rules_detailed,
}

display(pd.DataFrame([payload_detailed["totals"]]))
print("JSON FINAL DETALLADO:")
print(json.dumps(payload_detailed, ensure_ascii=False, indent=2))



,total_rules,failed_rules,error_rules,total_fallos
0,1,0,0,0


,mensaje
0,No hubo fallos


{
  "totals": {
    "rules": 1,
    "failed": 0,
    "errors": 0,
    "passed": 1,
    "with_no_aplica": 0
  },
  "failed_rules_detailed": [],
  "error_rules": []
}
